# SAC: Soft Actor-Critic

References: https://spinningup.openai.com/en/latest/algorithms/sac.html

## Install Dependencies

In [ ]:
!pip install gymnasium

## Helpers

In [ ]:
# Video management imports
import cv2
import matplotlib.pyplot as plt

# Check if we running in Google Colab or Jupyter Notebook
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print('Running in Google Colab')
    # Do you need to connect with Google Drive? Uncomment the next two lines
    # from google.colab import drive
    # drive.mount('/content/drive')
    # This auxiliary function simplifies the visualization of OpenCV Images
    from google.colab.patches import cv2_imshow
else:
    print('Running in Jupyter Notebook')
    # This auxiliary function simplifies the visualization of OpenCV Images
    def cv2_imshow(img, title=''):
        if img.ndim > 2:
            img= cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            # view the image in its natural size
            plt.figure(figsize=(img.shape[1], img.shape[0]), dpi=1)
            plt.imshow(img)
            plt.title(title)
            plt.xticks([]), plt.yticks([])
            plt.show()
        else:
            # view the image in its natural size
            plt.figure(figsize=(img.shape[1], img.shape[0]), dpi=1)
            plt.imshow(img, cmap='gray')
            plt.title(title)
            plt.xticks([]), plt.yticks([])
            plt.show()

# Helper functions to save videos and images
def save_video(img_array, path='/content/test.mp4'):
  height, width, layers = img_array[0].shape
  size = (width, height)
  out = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*'MP4V'), 15, size)
  if out.isOpened():
    for i in range(len(img_array)):
      bgr_img = cv2.cvtColor(img_array[i], cv2.COLOR_RGB2BGR)
      out.write(bgr_img)
    out.release()
    print('Video saved.')
  else:
    print(f'Could not save video path: {path}')


# SAC: Soft Actor-Critic

**Soft Actor-Critic (SAC)** is an off-policy, model-free reinforcement learning algorithm designed for continuous action spaces. It incorporates entropy maximization to encourage exploration by making the policy take diverse actions. SAC uses:

- **Two Critic Networks (Q-functions)**: To address overestimation bias.
- **Actor Network (Policy)**: Outputs continuous actions.
- **Entropy Regularization**: Controlled by temperature parameter α to balance exploration and exploitation.
- **Experience Replay Buffer**: Stores transitions to sample mini-batches from.



## Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F
from tqdm.notebook import tqdm
from collections import deque
import random

import gymnasium as gym

print(f"Gym version: {gym.__version__}")
print(f"Torch version: {torch.__version__}")

## Policy and Q-Networks
SAC uses three different types of networks:
- **Policy Network**: Outputs mean and log standard deviation of actions.
- **Two Q-Networks**: Estimate Q-values.
- **Target Q-Networks**: For stability in training (soft updates).

Both Q-Networks are implemented using the same NN architecture.


In [ ]:
class PolicyNetwork(nn.Module):
    def __init__(self, env, state_size, action_size, hidden_size=256):
        super().__init__()
        self.fc1 = nn.Linear(state_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.mean = nn.Linear(hidden_size, action_size)
        self.log_std = nn.Linear(hidden_size, action_size)
        # Register action_scale and action_bias as buffers to ensure they're moved to the correct device
        action_scale = (env.action_space.high - env.action_space.low) / 2.
        action_bias = (env.action_space.high + env.action_space.low) / 2.
        self.register_buffer('action_scale', torch.tensor(action_scale, dtype=torch.float32))
        self.register_buffer('action_bias', torch.tensor(action_bias, dtype=torch.float32))

    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        mean = self.mean(x)
        log_std = self.log_std(x)
        log_std = torch.clamp(log_std, min=-20, max=2)  # To avoid numerical issues
        return mean, log_std

    def select_action(self, state):
        mean, log_std = self(state)
        std = log_std.exp()
        normal = torch.distributions.Normal(mean, std)
        x_t = normal.rsample()  # Reparameterization trick
        y_t = torch.tanh(x_t)
        action = y_t * self.action_scale + self.action_bias
        # Correct log_prob calculation without including action_scale
        log_prob = normal.log_prob(x_t)
        log_prob -= torch.log(1 - y_t.pow(2) + 1e-6)
        log_prob = log_prob.sum(dim=1, keepdim=True)
        return action, log_prob

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, state_size, action_size, hidden_size=256):
        super().__init__()
        self.fc1 = nn.Linear(state_size + action_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.q = nn.Linear(hidden_size, 1)

    def forward(self, state, action):
        x = torch.cat([state, action], 1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        q_value = self.q(x)
        return q_value

## Replay Buffer
SAC is off-policy and requires an experience replay buffer.

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def add(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, BATCH_SIZE):
        state, action, reward, next_state, done = zip(*random.sample(self.buffer, BATCH_SIZE))
        return (np.array(state), np.array(action), np.array(reward), np.array(next_state), np.array(done))

    def size(self):
        return len(self.buffer)

## Agent Definition

The constructor of the agent performs the usual activities:
- Initialize policy network, two Q-networks, and target Q-networks.
- Define optimizers for each network.

In this case there are additional parameters:
- **Entropy Temperature $\alpha$**:
  - Automatically adjusted to maintain a target entropy, promoting exploration.
  - Initialized in the agent and updated during training.

- **Target update rate $\tau$**:
  - Used to merge the trained Q-network into the target network, performing a linear combination of the old and new network, instead of just subsituting the old network as in other methods.

## Loss functions

During training SAC optimizes two-different types of networks: the policy network and the two q-networks.

### Policy network loss function

The loss function for the **policy network** is designed to maximize both the expected return and the **entropy of the policy**. The goal is to encourage exploration by promoting a higher entropy policy (a less certain policy) while also improving the expected cumulative reward.

The policy loss function in SAC is derived from the maximum entropy reinforcement learning framework, and it is formulated as follows:

$
\mathcal{L}_{\pi} = \mathbb{E}_{s_t \sim \mathcal{D}} \left[ \mathbb{E}_{a_t \sim \pi(\cdot|s_t)} \left[ \alpha \log (\pi(a_t|s_t)) - Q(s_t, a_t) \right] \right]
$

Where:

- $ \mathcal{D} $ is the replay buffer containing past experiences.
- $ s_t $ is the state at time $ t $.
- $ a_t $ is the action sampled from the policy $ \pi $.
- $ \alpha $ is the temperature parameter controlling the trade-off between the entropy term and the reward. It can be a fixed hyperparameter or adjusted automatically during training.
- $ \log(\pi(a_t|s_t)) $ is the log probability of the action under the current policy, representing the entropy term.
- $ Q(s_t, a_t) $ is the Q-value estimated by the critic network for the state-action pair.

The policy loss aims to minimize the expected difference between the action-value $ Q(s_t, a_t) $ and the entropy-augmented reward $ \alpha \log (\pi(a_t|s_t)) $. This effectively encourages the policy to select actions that not only have high expected returns but also contribute to maintaining high entropy, thus fostering exploration.

The inclusion of the entropy term in the loss function is what differentiates SAC from traditional actor-critic methods, making it particularly effective for environments where exploration is crucial.

### Q-network loss function

The loss functions for the Q-networks are designed to minimize the difference between the predicted Q-values and the target Q-values. SAC typically employs two Q-networks to address the overestimation bias commonly encountered in value function estimation, a technique known as "Twin Q-learning."

The loss function for each Q-network is based on the mean squared error between the predicted Q-value and a target Q-value. The loss for a Q-network $ Q_{\theta_i} $ (where $ i $ could be 1 or 2 for the two Q-networks) is given by:

$
\mathcal{L}(\theta_i) = \mathbb{E}_{(s_t, a_t, r_t, s_{t+1}) \sim \mathcal{D}} \left[ \left( Q_{\theta_i}(s_t, a_t) - y_t \right)^2 \right]
$

Where:

- $ Q_{\theta_i}(s_t, a_t) $ is the predicted Q-value from the Q-network with parameters $ \theta_i $.
- $ y_t $ is the target Q-value, computed as:

$
y_t = r_t + \gamma \, \mathbb{E}_{a_{t+1} \sim \pi(\cdot|s_{t+1})} \left[ \min_{j=1,2} Q_{\bar{\theta}_j}(s_{t+1}, a_{t+1}) - \alpha \log \pi(a_{t+1}|s_{t+1}) \right]
$

Where:

- $ r_t $ is the reward received at time step $ t $.
- $ \gamma $ is the discount factor.
- $ Q_{\bar{\theta}_j} $ are the target Q-networks with parameters $\bar{\theta}_j$, which are typically a slowly moving average of the Q-network parameters $\theta_j$.
- $ \min_{j=1,2} Q_{\bar{\theta}_j}(s_{t+1}, a_{t+1}) $ uses the minimum of the two target Q-values to mitigate overestimation.
- $ \alpha \log \pi(a_{t+1}|s_{t+1}) $ is the entropy term, encouraging exploration by adding the expected entropy of the next action under the policy $\pi$.

This loss function ensures that the Q-networks are trained to accurately predict the expected return of actions while considering the exploration-exploitation trade-off facilitated by the entropy term. The use of two Q-networks and the minimum operator helps to stabilize learning and reduce overestimation bias.



In [ ]:
class SACAgent:
    def __init__(self, env, state_size, action_size, device, batch_size, gamma=0.99, tau=0.005, lr=3e-4):
        self.device = device
        self.replay_buffer = ReplayBuffer(capacity=1000000)
        self.batch_size = batch_size
        self.gamma = gamma
        self.tau = tau

        # Policy Network (Actor)
        self.policy = PolicyNetwork(env, state_size, action_size).to(device)
        self.policy_optimizer = optim.Adam(self.policy.parameters(), lr=lr)

        # Q-Networks (Critics)
        self.q1 = QNetwork(state_size, action_size).to(device)
        self.q1_optimizer = optim.Adam(self.q1.parameters(), lr=lr)

        self.q2 = QNetwork(state_size, action_size).to(device)
        self.q2_optimizer = optim.Adam(self.q2.parameters(), lr=lr)

        # Target Q-Networks
        self.q1_target = QNetwork(state_size, action_size).to(device)
        self.q2_target = QNetwork(state_size, action_size).to(device)
        self.q1_target.load_state_dict(self.q1.state_dict())
        self.q2_target.load_state_dict(self.q2.state_dict())

        self.target_entropy = -action_size  # For automatic alpha adjustment
        self.log_alpha = torch.zeros(1, requires_grad=True, device=device)
        self.alpha_optimizer = optim.Adam([self.log_alpha], lr=lr)
        # Initialize alpha
        self.alpha = self.log_alpha.exp().detach()

    def select_action(self, state):
        state = torch.FloatTensor(state).unsqueeze(0).to(self.device)
        action, _ = self.policy.select_action(state)
        return action.detach().cpu().numpy()[0]

    def train(self):
        if self.replay_buffer.size() < self.batch_size:
            return
        # Sample a batch from replay_buffer
        state_batch, action_batch, reward_batch, next_state_batch, done_batch = self.replay_buffer.sample(self.batch_size)

        state_batch = torch.FloatTensor(state_batch).to(self.device)
        action_batch = torch.FloatTensor(action_batch).to(self.device)
        reward_batch = torch.FloatTensor(reward_batch).unsqueeze(1).to(self.device)
        next_state_batch = torch.FloatTensor(next_state_batch).to(self.device)
        done_batch = torch.FloatTensor(np.float32(done_batch)).unsqueeze(1).to(self.device)

        # Update alpha
        self.alpha = self.log_alpha.exp()

        with torch.no_grad():
            next_action, next_log_prob = self.policy.select_action(next_state_batch)
            q1_next = self.q1_target(next_state_batch, next_action)
            q2_next = self.q2_target(next_state_batch, next_action)
            min_q_next = torch.min(q1_next, q2_next)
            q_target = reward_batch + (1 - done_batch) * self.gamma * (min_q_next - self.alpha * next_log_prob)

        # Q1 loss
        q1_current = self.q1(state_batch, action_batch)
        q1_loss = F.mse_loss(q1_current, q_target)

        # Q2 loss
        q2_current = self.q2(state_batch, action_batch)
        q2_loss = F.mse_loss(q2_current, q_target)

        # Update Q-functions
        self.q1_optimizer.zero_grad()
        q1_loss.backward()
        self.q1_optimizer.step()

        self.q2_optimizer.zero_grad()
        q2_loss.backward()
        self.q2_optimizer.step()

        # Policy loss
        pi, log_pi = self.policy.select_action(state_batch)
        q1_pi = self.q1(state_batch, pi)
        q2_pi = self.q2(state_batch, pi)
        min_q_pi = torch.min(q1_pi, q2_pi)
        policy_loss = (self.alpha * log_pi - min_q_pi).mean()

        self.policy_optimizer.zero_grad()
        policy_loss.backward()
        self.policy_optimizer.step()

        # Alpha loss (for entropy temperature adjustment)
        alpha_loss = -(self.log_alpha * (log_pi + self.target_entropy).detach()).mean()
        self.alpha_optimizer.zero_grad()
        alpha_loss.backward()
        self.alpha_optimizer.step()

        # Soft update of target networks
        for target_param, param in zip(self.q1_target.parameters(), self.q1.parameters()):
            target_param.data.copy_(target_param.data * (1.0 - self.tau) + param.data * self.tau)
        for target_param, param in zip(self.q2_target.parameters(), self.q2.parameters()):
            target_param.data.copy_(target_param.data * (1.0 - self.tau) + param.data * self.tau)

## Training

In [ ]:
NUM_EPISODES = 200
BATCH_SIZE = 256
MAX_TIMESTEPS = 200  # Max steps per episode
GAMMA = 0.99
TAU = 0.005
LEARNING_RATE = 3e-4
ENV_NAME = 'Pendulum-v1'  # Continuous action space

In [ ]:
device = torch.device("cpu")
env = gym.make(ENV_NAME)

In [ ]:
# Observation-space of Pendulum-v1 (3)
state_size = env.observation_space.shape[0]

# Action-space of Pendulum-v1 (1)
action_size = env.action_space.shape[0]

agent = SACAgent(env, state_size, action_size, device, batch_size = BATCH_SIZE, gamma=GAMMA, tau=TAU, lr=LEARNING_RATE)

In [ ]:
episode_rewards = []

for episode in range(NUM_EPISODES):
    state, _ = env.reset()
    episode_reward = 0
    episode_count = 0
    done = False
    while not done:
        action = agent.select_action(state)

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        agent.replay_buffer.add(state, action, reward, next_state, done)
        state = next_state
        episode_reward += reward

        agent.train()

        episode_count += 1

        if episode_count > MAX_TIMESTEPS:
            break

    episode_rewards.append(episode_reward)
    print(f"Episode: {episode}, Reward: {episode_reward}")

## Viewing Statistics

Let's visualize the accumulated reward during training:

In [ ]:
x = np.arange(len(episode_rewards))
# Plot the values
plt.plot(x, episode_rewards)
plt.xlabel('Episode')
plt.ylabel('Reward')
plt.title('SAC for Pendulum-v1')
plt.show()

## Testing the SAC Agent

Finally, let's visualize our agent in operation:

In [ ]:
env = gym.make(ENV_NAME, render_mode="rgb_array")

images = []
for _ in range(0,5):
  observation, _ = env.reset()
  done = False
  while not done:
      with torch.no_grad():
        action = agent.select_action(observation)

      observation, reward, terminated, truncated, _ = env.step(action)
      done = terminated or truncated
      image = env.render()
      images.append(image)

save_video(images, path='/content/SAC_pendulum.mp4')